# MetroRide with Prefect: run the lakehouse pipeline as a flow

The phase notebooks run *ingest → silver → gold* by hand, one cell at a time. `pipeline.py` turns the same
functions into **Prefect tasks** inside one **flow**, adds retries, a quality check, MLflow training, and a
schedule:

```
nightly_pipeline(month)
  ingest -> bronze_to_silver -> silver_to_gold -> check_gold -> train_and_log -> promote_if_better
```

| Step | What you see |
|---|---|
| 1 | One flow run: every task gets a state, a log and a timing |
| 2 | A crash in the middle: Prefect retries, and no row is written twice |
| 3 | The same run again by mistake: nothing changes |
| 4 | Backfill June–September: the flow trains and promotes models in MLflow |
| 5 | Read the run history back from Prefect and MLflow |
| 6 | Schedule the flow with a deployment (terminal) |

**Before you start** (from the `mlops/` folder)

1. `pip install -r requirements.txt`
2. Optional but recommended, in a separate terminal: `prefect server start` and open http://127.0.0.1:4200.
   Then, before starting Jupyter: `prefect config set PREFECT_API_URL=http://127.0.0.1:4200/api`.
   Without a server, Prefect starts a temporary one inside this kernel, and its UI disappears with the kernel.
3. Shut down `tutorial_mlflow.ipynb` first: the lakehouse catalog allows **one process at a time**.

Open `pipeline.py` next to this notebook: every task is a few lines around a function of `lakehouse.py`.

## 0. Setup: start from an empty lakehouse

In [ ]:
from prefect.settings import PREFECT_API_URL

import lakehouse as lh
import ml
from pipeline import nightly_pipeline, backfill

print("Prefect API:", PREFECT_API_URL.value() or "none: a temporary server will start in this kernel")
lh.reset_lakehouse()          # MLflow runs and registered models are kept

## 1. One flow run

Calling the flow function starts a **flow run**. Each task call inside it is a **task run** with its own
state (`Completed`, `Failed`, `AwaitingRetry`, ...) and log lines. April has no driver-app events, so
`train_and_log` skips training.

In [ ]:
nightly_pipeline("2026-04")

In [ ]:
lh.row_counts()

## 2. A crash in the middle, and a retry

`simulate_crash=True` makes `bronze_to_silver` fail **after** it has committed the new `silver.trips` rows and
**before** `silver.trip_context`. The task has `retries=2, retry_delay_seconds=30`, so Prefect waits 30 seconds
and runs it again.

Watch the log: the retry adds **0** trips (the `NOT EXISTS` anti-join finds them already in silver) and the
137 missing `trip_context` rows. Retries are only safe because every step is idempotent.

In [ ]:
nightly_pipeline("2026-05", simulate_crash=True)

In [ ]:
counts_after_may = lh.row_counts()
snapshots_after_may = lh.sql("SELECT max(snapshot_id) AS id FROM metroride.snapshots()").id[0]
counts_after_may

## 3. The same run again, by mistake

The file guard skips both files, the anti-joins find nothing new, and gold has no new dates. No table grows,
and no DuckLake snapshot is created, because nothing is written.

In [ ]:
nightly_pipeline("2026-05")

same_counts = lh.row_counts().n_rows.equals(counts_after_may.n_rows)
same_snapshot = lh.sql("SELECT max(snapshot_id) AS id FROM metroride.snapshots()").id[0] == snapshots_after_may
print("row counts unchanged:", same_counts, "| no new snapshot:", same_snapshot)

## 4. Backfill June to September

`backfill` is a flow that calls `nightly_pipeline` once per month, so each month is a **subflow run**.
From July on there is a previous month to train on, so `train_and_log` logs an MLflow run (tagged with the
DuckLake snapshot) and `promote_if_better` moves the `@champion` alias when the test MAE improves.
Spark starts on the first training run, so this takes a minute or two.

In [ ]:
backfill(["2026-06", "2026-07", "2026-08", "2026-09"])

## 5. Read the history back

**MLflow**: the nightly runs, their snapshot tags, and the current champion.

In [ ]:
import mlflow
from mlflow import MlflowClient

ml.setup_mlflow("metroride-nightly")
runs = mlflow.search_runs(experiment_names=["metroride-nightly"], order_by=["start_time ASC"])
display(runs[["tags.mlflow.runName", "params.test_month", "metrics.mae", "tags.ducklake.snapshot_id"]].round(3))

champion = MlflowClient().get_model_version_by_alias(ml.REGISTERED_MODEL, "champion")
print(f"@champion -> {ml.REGISTERED_MODEL} version {champion.version} (run {champion.run_id[:8]})")

Note that each nightly MAE comes from a **different test month**. The September model is not promoted
although it is the same model as Model 2 in the Spark ML tutorial: comparing MAEs across test sets is not fair
(exercise 3).

**Prefect**: the same run history the UI shows, read through the API.

In [ ]:
import pandas as pd
from prefect.client.orchestration import get_client
from prefect.client.schemas.sorting import FlowRunSort

with get_client(sync_client=True) as client:
    flow_runs = client.read_flow_runs(sort=FlowRunSort.START_TIME_DESC, limit=10)
pd.DataFrame([{"flow run": r.name, "month": r.parameters.get("month"), "state": r.state_name,
               "seconds": round(r.total_run_time.total_seconds(), 1)} for r in flow_runs])

**Gold**: the rainy vs. dry comparison of phase 3, now produced by the flow.

In [ ]:
lh.sql('''
    SELECT CASE WHEN is_rainy THEN 'rainy' ELSE 'dry' END AS weather,
           sum(requests) AS requests, sum(completed_trips) AS completed, sum(cancelled_trips) AS cancelled,
           round(100.0 * sum(cancelled_trips) / sum(requests), 2) AS cancellation_rate_pct,
           round(sum(average_driver_arrival_wait_minutes * completed_trips) / sum(completed_trips), 2)
               AS avg_driver_arrival_wait
    FROM metroride.gold.daily_service_performance
    WHERE strftime(service_date, '%Y-%m') = '2026-09'
    GROUP BY ALL ORDER BY weather
''')

## 6. Schedule it: a deployment

A notebook is not a scheduler. In a terminal, in the `mlops/` folder (with `prefect server start` running
and this kernel shut down):

```
python pipeline.py serve
```

`serve()` registers the deployment **`metroride-nightly/nightly`** with the schedule
`Cron("0 2 * * *", timezone="Asia/Bangkok")` and keeps running, starting a flow run every night at 02:00.
From a second terminal, or the **Run** button in the UI, start a run now with other parameters:

```
prefect deployment run "metroride-nightly/nightly" -p month=2026-06
```

`python pipeline.py run 2026-05 --crash` and `python pipeline.py backfill` run the same flows without a
deployment. In production, `serve()` is replaced by work pools and workers (Docker, Kubernetes, cloud).

## Exercises

1. Extend `check_gold` in `pipeline.py`: for September, fail the flow if the rainy cancellation rate is not
   11.11% or the dry one is not 4.17% (expected values in the phase-3 README).
2. Change `promote_if_better` so that it promotes only when the MAE improves by at least 0.05 minutes.
3. Make the comparison fair: score the current champion on the **same** test month as the new model before
   deciding. Is the September model promoted now?
4. Set `retries=0` on `bronze_to_silver` and run the crash drill again. What state does the flow end in, and
   what does the next normal run do?

## Clean up

In [ ]:
lh.close_catalog()     # release the catalog for other processes (pipeline.py, tutorial_mlflow.ipynb)